![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 10 -- Lab 2: Image Search with Pretrained Embeddings

Google Images, Pinterest, and e-commerce sites all use the same core idea: convert images into **embedding vectors** using a pretrained CNN, then find the **nearest neighbors** in that vector space.

In this lab you will build a working image search engine from scratch.

| Part | Goal |
|---|---|
| Setup & Data | Load Caltech-101, create DataLoader |
| Feature Extraction | Use EfficientNetV2-S as a feature extractor |
| Compute Embeddings | Extract embeddings for 1000 images |
| PCA (given) | Compress embeddings for faster search |
| Image Search | Find similar images with cosine similarity |
| Compare | Raw vs PCA-reduced search quality and speed |

**Key takeaway:** Pretrained CNNs produce embeddings where visually similar images are close together -- no task-specific training needed.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset

import numpy as np
import matplotlib.pyplot as plt
import random
import time
import os

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from tqdm import tqdm

plt.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
# Part 1 -- Setup and Data (GIVEN)

We use the **Caltech-101** dataset, which contains images from 101 object categories (faces, airplanes, animals, etc.). This variety makes it ideal for testing image search.

In [ ]:
# --- GIVEN: Download Caltech-101 ---
!echo "Downloading Caltech-101 dataset..."
!curl -L -o 101_ObjectCategories.zip --progress-bar https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip?download=1
!unzip -q 101_ObjectCategories.zip
!tar -xzf caltech-101/101_ObjectCategories.tar.gz
!rm -f 101_ObjectCategories.zip
!echo "Done! Categories:"
!ls 101_ObjectCategories | head -20

In [ ]:
# --- GIVEN: Create dataset and DataLoader ---
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

dataset_path = '101_ObjectCategories'
full_dataset = ImageFolder(root=dataset_path, transform=transform)

n_images = 1000
subset_indices = random.sample(range(len(full_dataset)), n_images)
dataset = Subset(full_dataset, subset_indices)

batch_size = 64
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

class_names = full_dataset.classes
print(f'Total images in Caltech-101: {len(full_dataset)}')
print(f'Subset size: {len(dataset)}')
print(f'Number of categories: {len(class_names)}')
print(f'Batches: {len(data_loader)}')

In [ ]:
# --- GIVEN: Visualize sample images ---
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    img, label = dataset[i]
    img_np = img.permute(1, 2, 0).numpy()
    img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_np = np.clip(img_np, 0, 1)
    real_label = full_dataset.targets[subset_indices[i]]
    ax.imshow(img_np)
    ax.set_title(class_names[real_label], fontsize=9)
    ax.axis('off')

fig.suptitle('Sample Images from Caltech-101 Subset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# Part 2 -- Load Pretrained Model as Feature Extractor

A pretrained CNN has two parts:
- **Feature extractor** (convolutional backbone): learns visual patterns.
- **Classification head** (final FC layers): maps features to ImageNet class labels.

We only need the feature extractor. Its output -- before the classification head -- is the **embedding**.

## Task 1: Load EfficientNetV2-S with Pretrained Weights

Fill in the `None` to load the model with ImageNet-pretrained weights and set it to evaluation mode.

In [ ]:
efficientnet = models.efficientnet_v2_s(
    weights=None  # TODO: use models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
)
efficientnet.eval().to(device)

print(efficientnet)

## Task 2: Write the Feature Extraction Function

We pass images through `model.features` (the convolutional backbone), then apply **adaptive average pooling** to get a fixed-size vector regardless of input spatial dimensions, and **flatten** it.

Fill in the `None` values:
- `nn.AdaptiveAvgPool2d` with output size `(1, 1)` collapses each feature map to a single value.
- `torch.flatten` with `start_dim=1` converts from `[batch, channels, 1, 1]` to `[batch, channels]`.

In [ ]:
pool = nn.AdaptiveAvgPool2d(None)  # TODO: output spatial size (1, 1)

def extract_features(image_tensor):
    image_tensor = image_tensor.to(device)
    with torch.no_grad():
        features = efficientnet.features(image_tensor)
        features = pool(features)
        features = torch.flatten(features, start_dim=None)  # TODO: which dimension to start flattening?
    return features.cpu().numpy()

## Task 3: Test on One Batch

Verify that the feature extraction works and check the embedding dimension.

Fill in the `None` to call your `extract_features` function.

In [ ]:
sample_batch, _ = next(iter(data_loader))
sample_embeddings = None  # TODO: call extract_features on sample_batch

print(f'Batch shape:     {sample_batch.shape}')
print(f'Embedding shape: {sample_embeddings.shape}')
print(f'Embedding dim:   {sample_embeddings.shape[1]}')

---
# Part 3 -- Compute Embeddings for All Images

## Task 4: Extract Features for the Full Subset

Loop over the DataLoader and build a matrix of all embeddings.

Fill in the `None` to call your `extract_features` function inside the loop.

In [ ]:
all_features = []
all_labels = []

for images, labels in tqdm(data_loader, desc='Extracting features'):
    features = None  # TODO: call extract_features on this batch
    all_features.append(features)
    all_labels.extend(labels.numpy())

all_features = np.concatenate(all_features, axis=0)
all_labels = np.array(all_labels)

print(f'Embeddings matrix shape: {all_features.shape}')  # should be [1000, 1280]
print(f'Labels array shape:      {all_labels.shape}')

---
# Part 4 -- PCA Dimensionality Reduction (GIVEN)

Each embedding is a 1280-dimensional vector. Searching through high-dimensional vectors can be slow for large databases. **PCA (Principal Component Analysis)** compresses these vectors to a lower dimension while preserving most of the useful information.

Think of it as: "Keep only the 128 most important directions in the 1280-d space."

You will learn the math behind PCA later -- for now, just observe how it affects search quality and speed.

In [ ]:
# --- GIVEN: Apply PCA to reduce from 1280-d to 128-d ---
n_pca_components = 128

pca = PCA(n_components=n_pca_components)
pca_features = pca.fit_transform(all_features)

explained = sum(pca.explained_variance_ratio_) * 100
print(f'PCA: {all_features.shape[1]}d -> {n_pca_components}d')
print(f'Variance retained: {explained:.1f}%')
print(f'PCA embeddings shape: {pca_features.shape}')

---
# Part 5 -- Image Search with Cosine Similarity

Given a query image, we find the most similar images by computing the **cosine similarity** between the query embedding and all stored embeddings:

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}$$

Cosine similarity ranges from -1 to 1. Higher values mean more similar.

## Task 5: Implement the Search Function

Fill in the `None` values:
1. Compute cosine similarity between the query embedding and **all** embeddings.
2. Sort by similarity (highest first) and return the top-k indices.

**Hint:** `cosine_similarity` from sklearn expects 2D arrays. The query is shape `(1280,)` so reshape it to `(1, 1280)` using `.reshape(1, -1)`.

In [ ]:
def search_similar(query_idx, embeddings, top_k=5):
    """Find the top_k most similar images to the query image.
    
    Args:
        query_idx: index of the query image in the dataset
        embeddings: matrix of shape [n_images, embedding_dim]
        top_k: number of results to return
    
    Returns:
        indices: array of top_k most similar image indices (excluding the query itself)
        scores: array of corresponding similarity scores
    """
    query_vector = embeddings[query_idx]
    
    similarities = cosine_similarity(
        None,       # TODO: reshape query_vector to (1, embedding_dim)
        embeddings
    ).flatten()
    
    # Sort by similarity, highest first
    ranked_indices = np.argsort(None)[::-1]  # TODO: what to sort?
    
    # Remove the query image itself from results
    ranked_indices = ranked_indices[ranked_indices != query_idx]
    
    top_indices = ranked_indices[:top_k]
    top_scores  = similarities[top_indices]
    
    return top_indices, top_scores

In [ ]:
# --- GIVEN: Visualization helper ---
def show_search_results(query_idx, indices, scores, title_prefix=''):
    """Display the query image and top-k retrieved images."""
    n_results = len(indices)
    fig, axes = plt.subplots(1, n_results + 1, figsize=(3 * (n_results + 1), 3.5))

    # Show query
    img, _ = dataset[query_idx]
    img_np = img.permute(1, 2, 0).numpy()
    img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_np = np.clip(img_np, 0, 1)
    real_label = full_dataset.targets[subset_indices[query_idx]]
    axes[0].imshow(img_np)
    axes[0].set_title(f'QUERY\n{class_names[real_label]}', fontsize=10, fontweight='bold', color='#8c1515')
    axes[0].axis('off')

    # Show results
    for j, (idx, score) in enumerate(zip(indices, scores)):
        img, _ = dataset[idx]
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)
        real_label = full_dataset.targets[subset_indices[idx]]
        axes[j+1].imshow(img_np)
        axes[j+1].set_title(f'{class_names[real_label]}\nsim={score:.3f}', fontsize=9)
        axes[j+1].axis('off')

    fig.suptitle(f'{title_prefix}Image Search Results', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Task 6: Run Search Queries

Pick 3 different query images and find the top-5 most similar images using the **PCA-reduced** embeddings.

Fill in the `None` values to call your `search_similar` function.

In [ ]:
query_indices = [0, 100, 500]

for q_idx in query_indices:
    indices, scores = None  # TODO: call search_similar with pca_features, top_k=5
    show_search_results(q_idx, indices, scores)

---
# Part 6 -- Compare Raw vs PCA Embeddings

Does compressing from 1280-d to 128-d hurt search quality? Let's compare.

## Task 7: Side-by-Side Comparison

Run the same queries on both the **raw** (1280-d) and **PCA** (128-d) embeddings.

Fill in the `None` values.

In [ ]:
query_idx = 0

# Search with raw embeddings
raw_indices, raw_scores = None  # TODO: call search_similar with all_features
show_search_results(query_idx, raw_indices, raw_scores, title_prefix='Raw (1280-d) -- ')

# Search with PCA embeddings
pca_indices, pca_scores = None  # TODO: call search_similar with pca_features
show_search_results(query_idx, pca_indices, pca_scores, title_prefix='PCA (128-d) -- ')

In [ ]:
# --- GIVEN: Timing comparison ---
n_runs = 100

start = time.time()
for _ in range(n_runs):
    cosine_similarity(all_features[0:1], all_features)
raw_time = (time.time() - start) / n_runs * 1000

start = time.time()
for _ in range(n_runs):
    cosine_similarity(pca_features[0:1], pca_features)
pca_time = (time.time() - start) / n_runs * 1000

print(f'Average search time (raw 1280-d): {raw_time:.2f} ms')
print(f'Average search time (PCA  128-d): {pca_time:.2f} ms')
print(f'Speedup: {raw_time/pca_time:.1f}x')

---
# Bonus: Try a Different Model (Optional)

Swap out EfficientNetV2-S for **ResNet-18** and compare the search results. ResNet-18 produces 512-d embeddings instead of 1280-d.

**Hint:** For ResNet, the backbone is everything except `model.fc`. You can use:
```python
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet.fc = nn.Identity()  # removes the classification head
```

In [ ]:
# Your code here
